# CertusDoc - Multi-Agent Document Forgery Detection
## Secure AI Hackathon 2026 | IIT Madras x BITS Pilani Goa

**Team:** Blue Team (Defense)

CertusDoc is a 4-stage multi-agent pipeline for detecting document forgery:
1. **Ingestion** - PDF/image intake, 300 DPI rendering, Tesseract OCR
2. **Multi-Agent Detection** (parallel) - Visual Tamper Agent (ManTraNet + ELA), Text Forensics Agent, Metadata & Provenance Agent
3. **Weighted Trust Fusion** - DIS = \u03a3(R\u1d62 \u00d7 S\u1d62) / \u03a3R\u1d62 with dynamic reliability weights
4. **Forensic Report** - PDF report with heatmaps, per-agent findings, risk classification

**DIS (Document Integrity Score):** 0.0 = Forged, 1.0 = Authentic

| Risk Level | DIS Range |
|---|---|
| AUTHENTIC | >= 0.80 |
| LOW RISK | 0.65 - 0.80 |
| MEDIUM RISK | 0.40 - 0.65 |
| HIGH RISK | < 0.40 |


---
## 1. Setup & Installation


In [ ]:
# Check GPU availability
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1024**3:.1f} GB")
else:
    print("WARNING: No GPU detected. Go to Runtime > Change runtime type > T4 GPU")


In [ ]:
# Clone the repository
!git clone https://github.com/nishantr14/Certus.git
%cd Certus/certusdoc


In [ ]:
# Install dependencies
!pip install -q -r requirements.txt

# Install system dependencies for OCR and PDF rendering
!apt-get install -qq tesseract-ocr poppler-utils libzbar0 > /dev/null 2>&1
print("Dependencies installed.")


In [ ]:
# Enable ManTraNet (GPU is available on Colab)
import os
os.environ["CERTUS_MANTRANET"] = "1"

# Import CertusDoc
import sys
sys.path.insert(0, ".")

from certusdoc.pipeline import CertusDocPipeline
from certusdoc.models import RiskLevel, ForgeryType, ForensicReport
from certusdoc.report.generator import generate_report

# Initialize pipeline
pipeline = CertusDocPipeline()
print("\nCertusDoc pipeline ready.")


---
## 2. Helper Functions for Visualization


In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from pathlib import Path
from IPython.display import display, HTML, Image as IPImage
import time


def get_risk_color(risk_level):
    """Return color for risk level."""
    colors = {
        RiskLevel.AUTHENTIC: "#2ecc71",
        RiskLevel.LOW_RISK: "#f39c12",
        RiskLevel.MEDIUM_RISK: "#e67e22",
        RiskLevel.HIGH_RISK: "#e74c3c",
    }
    return colors.get(risk_level, "#95a5a6")


def visualize_result(report, title=None):
    """Visualize a single document analysis result with heatmap."""
    fig, axes = plt.subplots(1, 3, figsize=(18, 6))

    # Original image
    if report.document.pages:
        page = report.document.pages[0]
        axes[0].imshow(cv2.cvtColor(page, cv2.COLOR_BGR2RGB))
    axes[0].set_title("Original Document", fontsize=12, fontweight="bold")
    axes[0].axis("off")

    # Heatmap
    colored_hm = None
    if report.fused_heatmap is not None:
        hm = report.fused_heatmap
        if hm.dtype != np.uint8:
            hm = (np.clip(hm, 0, 1) * 255).astype(np.uint8)
        colored_hm = cv2.applyColorMap(hm, cv2.COLORMAP_JET)
        axes[1].imshow(cv2.cvtColor(colored_hm, cv2.COLOR_BGR2RGB))
    axes[1].set_title("Forgery Heatmap", fontsize=12, fontweight="bold")
    axes[1].axis("off")

    # Overlay
    if report.document.pages and colored_hm is not None:
        page_rgb = cv2.cvtColor(report.document.pages[0], cv2.COLOR_BGR2RGB)
        hm_resized = cv2.resize(colored_hm, (page_rgb.shape[1], page_rgb.shape[0]))
        overlay = cv2.addWeighted(page_rgb, 0.6, cv2.cvtColor(hm_resized, cv2.COLOR_BGR2RGB), 0.4, 0)
        axes[2].imshow(overlay)
    axes[2].set_title("Overlay", fontsize=12, fontweight="bold")
    axes[2].axis("off")

    color = get_risk_color(report.risk_level)
    title_text = title or report.document.file_name
    fig.suptitle(
        f"{title_text}\n"
        f"DIS: {report.dis_score:.4f} | {report.risk_level.value} | "
        f"Forgery: {report.primary_forgery_type.value} | "
        f"Time: {report.processing_time_ms:.0f}ms",
        fontsize=13, fontweight="bold", color=color, y=1.02
    )
    plt.tight_layout()
    plt.show()


def print_agent_breakdown(report):
    """Print per-agent score breakdown."""
    display(HTML(f"<h4>Per-Agent Breakdown for {report.document.file_name}</h4>"))

    html = '<table style="border-collapse:collapse; width:100%; font-family:monospace;">'
    html += '<tr style="background:#1a1a2e; color:#0ff;">'
    html += '<th style="padding:8px; text-align:left;">Agent</th>'
    html += '<th style="padding:8px;">Score</th>'
    html += '<th style="padding:8px;">Weight</th>'
    html += '<th style="padding:8px;">Time</th>'
    html += '<th style="padding:8px; text-align:left;">Key Findings</th></tr>'

    for r in report.agent_results:
        score_color = "#2ecc71" if r.score >= 0.7 else "#e67e22" if r.score >= 0.4 else "#e74c3c"
        findings_text = "; ".join([f.description[:100] for f in r.findings[:3]])
        if len(r.findings) > 3:
            findings_text += f" (+{len(r.findings)-3} more)"

        html += f'<tr style="border-bottom:1px solid #333;">'
        html += f'<td style="padding:8px; font-weight:bold;">{r.agent_name}</td>'
        html += f'<td style="padding:8px; text-align:center; color:{score_color}; font-weight:bold;">{r.score:.4f}</td>'
        html += f'<td style="padding:8px; text-align:center;">{r.reliability_weight:.2f}</td>'
        html += f'<td style="padding:8px; text-align:center;">{r.processing_time_ms:.0f}ms</td>'
        html += f'<td style="padding:8px; font-size:11px;">{findings_text or "No findings"}</td></tr>'

    html += '</table>'
    display(HTML(html))


---
## 3. Analyze Authentic Documents
These are legitimate documents that should score **DIS >= 0.65** (AUTHENTIC or LOW RISK).


In [ ]:
authentic_dir = Path("data/authentic")
authentic_files = sorted(authentic_dir.glob("*"))
print(f"Found {len(authentic_files)} authentic documents:")
for f in authentic_files:
    print(f"  - {f.name} ({f.stat().st_size / 1024:.1f} KB)")


In [ ]:
authentic_reports = []

for f in authentic_files:
    if f.suffix.lower() in (".pdf", ".png", ".jpg", ".jpeg", ".tiff", ".tif", ".bmp"):
        print(f"\nAnalyzing: {f.name}")
        print("-" * 60)
        report = pipeline.analyze(str(f))
        authentic_reports.append(report)

        visualize_result(report, title=f"AUTHENTIC: {f.name}")
        print_agent_breakdown(report)
        print(f"\nRecommendation: {report.recommended_action}")
        print("=" * 80)

print(f"\n\nProcessed {len(authentic_reports)} authentic documents.")
auth_scores = [r.dis_score for r in authentic_reports]
if auth_scores:
    print(f"DIS range: [{min(auth_scores):.4f} - {max(auth_scores):.4f}]  avg={np.mean(auth_scores):.4f}")
    passed = sum(1 for s in auth_scores if s >= 0.65)
    print(f"Correctly identified as authentic (DIS >= 0.65): {passed}/{len(auth_scores)}")


---
## 4. Analyze Forged Documents
These are tampered documents that should score **DIS < 0.65** (MEDIUM or HIGH RISK).


In [ ]:
forged_dir = Path("data/forged")
forged_files = sorted(forged_dir.glob("*"))
print(f"Found {len(forged_files)} forged documents:")
for f in forged_files:
    print(f"  - {f.name} ({f.stat().st_size / 1024:.1f} KB)")


In [ ]:
forged_reports = []

for f in forged_files:
    if f.suffix.lower() in (".pdf", ".png", ".jpg", ".jpeg", ".tiff", ".tif", ".bmp"):
        print(f"\nAnalyzing: {f.name}")
        print("-" * 60)
        report = pipeline.analyze(str(f))
        forged_reports.append(report)

        visualize_result(report, title=f"FORGED: {f.name}")
        print_agent_breakdown(report)
        print(f"\nRecommendation: {report.recommended_action}")
        print("=" * 80)

print(f"\n\nProcessed {len(forged_reports)} forged documents.")
forg_scores = [r.dis_score for r in forged_reports]
if forg_scores:
    print(f"DIS range: [{min(forg_scores):.4f} - {max(forg_scores):.4f}]  avg={np.mean(forg_scores):.4f}")
    caught = sum(1 for s in forg_scores if s < 0.65)
    print(f"Correctly identified as forged (DIS < 0.65): {caught}/{len(forg_scores)}")


---
## 5. NaviDoMaSs Academic Dataset
Testing on the **NaviDoMaSs** (Navigation Document Manipulation) research dataset with genuine and forged payslip documents.

This is a published academic benchmark dataset containing:
- **Genuine** payslips (ground truth: authentic)
- **CopyPaste_Inter** - Cross-document copy-paste forgery
- **CopyPaste_Intra** - Within-document copy-paste forgery
- **Imitation** - Handwriting/content imitation forgery


In [ ]:
navidomass_dir = Path("data/navidomass/sample_dataset")
navidomass_reports = []

if navidomass_dir.exists():
    # Genuine documents
    genuine_dir = navidomass_dir / "Genuine"
    if genuine_dir.exists():
        print("=== GENUINE PAYSLIPS ===")
        for f in sorted(genuine_dir.glob("*.tif")):
            print(f"\nAnalyzing: {f.name}")
            report = pipeline.analyze(str(f))
            navidomass_reports.append(("genuine", report))
            visualize_result(report, title=f"GENUINE: {f.name}")

    # Forged documents (CopyPaste_Inter, CopyPaste_Intra, Imitation)
    for forgery_type in ["CopyPaste_Inter", "CopyPaste_Intra", "Imitation"]:
        forgery_dir = navidomass_dir / forgery_type
        if forgery_dir.exists():
            print(f"\n=== FORGED: {forgery_type.upper()} ===")
            tif_files = list(forgery_dir.rglob("*.tif"))
            # Filter out __MACOSX files
            tif_files = [f for f in tif_files if "__MACOSX" not in str(f)]
            for f in sorted(tif_files):
                print(f"\nAnalyzing: {f.name}")
                report = pipeline.analyze(str(f))
                navidomass_reports.append((forgery_type, report))
                visualize_result(report, title=f"FORGED ({forgery_type}): {f.name}")
else:
    print("NaviDoMaSs dataset not found. Skipping.")


In [ ]:
# NaviDoMaSs Summary
if navidomass_reports:
    genuine_scores = [r.dis_score for label, r in navidomass_reports if label == "genuine"]
    forged_navi = [(label, r) for label, r in navidomass_reports if label != "genuine"]
    forged_navi_scores = [r.dis_score for _, r in forged_navi]

    print("=" * 80)
    print("NAVIDOMASS DATASET SUMMARY")
    print("=" * 80)
    print(f"\nGenuine documents:  {len(genuine_scores)}")
    if genuine_scores:
        print(f"  DIS range: [{min(genuine_scores):.4f} - {max(genuine_scores):.4f}]")
        print(f"  Average DIS: {np.mean(genuine_scores):.4f}")
        print(f"  Correctly identified: {sum(1 for s in genuine_scores if s >= 0.65)}/{len(genuine_scores)}")

    print(f"\nForged documents:   {len(forged_navi_scores)}")
    if forged_navi_scores:
        print(f"  DIS range: [{min(forged_navi_scores):.4f} - {max(forged_navi_scores):.4f}]")
        print(f"  Average DIS: {np.mean(forged_navi_scores):.4f}")
        print(f"  Correctly detected: {sum(1 for s in forged_navi_scores if s < 0.65)}/{len(forged_navi_scores)}")

    # Breakdown by forgery type
    print("\n  Per-forgery-type breakdown:")
    for ftype in ["CopyPaste_Inter", "CopyPaste_Intra", "Imitation"]:
        scores = [r.dis_score for label, r in forged_navi if label == ftype]
        if scores:
            detected = sum(1 for s in scores if s < 0.65)
            print(f"    {ftype:20s}: avg DIS={np.mean(scores):.4f}, detected={detected}/{len(scores)}")

    # Combined accuracy
    total = len(genuine_scores) + len(forged_navi_scores)
    correct = sum(1 for s in genuine_scores if s >= 0.65) + sum(1 for s in forged_navi_scores if s < 0.65)
    print(f"\n  Overall NaviDoMaSs accuracy: {correct}/{total} = {correct/total:.1%}" if total else "")
else:
    print("No NaviDoMaSs results to summarize.")


---
## 6. Full Evaluation Metrics
Running the complete evaluation pipeline with confusion matrix, precision, recall, F1, and multi-threshold analysis.


In [ ]:
from eval import discover_files, run_evaluation, EvalSummary

# Discover all test files
files = discover_files("data")
print(f"Total evaluation files: {len(files)}")
print(f"  Authentic: {sum(1 for _, l in files if l == 'authentic')}")
print(f"  Forged: {sum(1 for _, l in files if l == 'forged')}")

# Run evaluation
threshold = 0.65
print(f"\nDecision threshold: {threshold}")
print("=" * 80)
summary = run_evaluation(pipeline, files, threshold=threshold)


In [ ]:
# Display evaluation metrics
display(HTML(f"""
<div style="background:#0d1117; color:#c9d1d9; padding:20px; border-radius:10px; font-family:monospace;">
<h2 style="color:#58a6ff; text-align:center;">CertusDoc Evaluation Results</h2>
<table style="width:60%; margin:auto; border-collapse:collapse;">
<tr style="border-bottom:2px solid #30363d;">
  <td style="padding:10px; font-size:16px;">Accuracy</td>
  <td style="padding:10px; font-size:20px; font-weight:bold; color:#2ecc71; text-align:right;">{summary.accuracy:.2%}</td>
</tr>
<tr style="border-bottom:2px solid #30363d;">
  <td style="padding:10px; font-size:16px;">Precision</td>
  <td style="padding:10px; font-size:20px; font-weight:bold; color:#2ecc71; text-align:right;">{summary.precision:.2%}</td>
</tr>
<tr style="border-bottom:2px solid #30363d;">
  <td style="padding:10px; font-size:16px;">Recall</td>
  <td style="padding:10px; font-size:20px; font-weight:bold; color:#2ecc71; text-align:right;">{summary.recall:.2%}</td>
</tr>
<tr style="border-bottom:2px solid #30363d;">
  <td style="padding:10px; font-size:16px;">F1 Score</td>
  <td style="padding:10px; font-size:20px; font-weight:bold; color:#58a6ff; text-align:right;">{summary.f1:.4f}</td>
</tr>
<tr style="border-bottom:2px solid #30363d;">
  <td style="padding:10px; font-size:16px;">Avg Authentic DIS</td>
  <td style="padding:10px; font-size:20px; font-weight:bold; color:#2ecc71; text-align:right;">{summary.avg_authentic_score:.4f}</td>
</tr>
<tr>
  <td style="padding:10px; font-size:16px;">Avg Forged DIS</td>
  <td style="padding:10px; font-size:20px; font-weight:bold; color:#e74c3c; text-align:right;">{summary.avg_forged_score:.4f}</td>
</tr>
</table>
</div>
"""))


In [ ]:
# Confusion Matrix Visualization
fig, ax = plt.subplots(1, 1, figsize=(6, 5))

cm = np.array([
    [summary.true_negatives, summary.false_positives],
    [summary.false_negatives, summary.true_positives]
])

im = ax.imshow(cm, interpolation='nearest', cmap='Blues')
ax.set_title("Confusion Matrix", fontsize=14, fontweight="bold")
ax.set_ylabel("True Label", fontsize=12)
ax.set_xlabel("Predicted Label", fontsize=12)
ax.set_xticks([0, 1])
ax.set_yticks([0, 1])
ax.set_xticklabels(["Authentic", "Forged"])
ax.set_yticklabels(["Authentic", "Forged"])

for i in range(2):
    for j in range(2):
        text_color = "white" if cm[i, j] > cm.max() / 2 else "black"
        ax.text(j, i, str(cm[i, j]), ha="center", va="center",
                fontsize=20, fontweight="bold", color=text_color)

plt.colorbar(im, ax=ax)
plt.tight_layout()
plt.show()

print(f"\nTP (forged correctly detected):     {summary.true_positives}")
print(f"TN (authentic correctly passed):    {summary.true_negatives}")
print(f"FP (authentic incorrectly flagged): {summary.false_positives}")
print(f"FN (forged incorrectly passed):     {summary.false_negatives}")


---
## 7. Score Distribution & Separation Analysis


In [ ]:
# Score distribution plot
all_reports = authentic_reports + forged_reports
auth_scores = [r.dis_score for r in authentic_reports]
forg_scores = [r.dis_score for r in forged_reports]

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Bar chart of all scores
names = [r.document.file_name[:25] for r in all_reports]
scores = [r.dis_score for r in all_reports]
bar_colors = ["#2ecc71" if r in authentic_reports else "#e74c3c" for r in all_reports]

bars = axes[0].barh(range(len(names)), scores, color=bar_colors)
axes[0].set_yticks(range(len(names)))
axes[0].set_yticklabels(names, fontsize=8)
axes[0].set_xlabel("DIS Score", fontsize=12)
axes[0].set_title("DIS Scores: All Documents", fontsize=13, fontweight="bold")
axes[0].axvline(x=0.65, color="orange", linestyle="--", linewidth=2, label="Threshold (0.65)")
axes[0].set_xlim(0, 1)
axes[0].legend()

# Add score labels on bars
for i, (score, bar) in enumerate(zip(scores, bars)):
    axes[0].text(min(score + 0.02, 0.92), i, f"{score:.3f}", va="center", fontsize=8, fontweight="bold")

# Score distribution histogram
if auth_scores:
    axes[1].hist(auth_scores, bins=10, alpha=0.7, color="#2ecc71", label=f"Authentic (n={len(auth_scores)})")
if forg_scores:
    axes[1].hist(forg_scores, bins=10, alpha=0.7, color="#e74c3c", label=f"Forged (n={len(forg_scores)})")
axes[1].axvline(x=0.65, color="orange", linestyle="--", linewidth=2, label="Threshold")
axes[1].set_xlabel("DIS Score", fontsize=12)
axes[1].set_ylabel("Count", fontsize=12)
axes[1].set_title("Score Distribution", fontsize=13, fontweight="bold")
axes[1].legend()

plt.tight_layout()
plt.show()

# Print separation stats
if auth_scores and forg_scores:
    gap = np.mean(auth_scores) - np.mean(forg_scores)
    print(f"\nScore Separation Analysis:")
    print(f"  Authentic: min={min(auth_scores):.4f}, max={max(auth_scores):.4f}, avg={np.mean(auth_scores):.4f}, std={np.std(auth_scores):.4f}")
    print(f"  Forged:    min={min(forg_scores):.4f}, max={max(forg_scores):.4f}, avg={np.mean(forg_scores):.4f}, std={np.std(forg_scores):.4f}")
    print(f"  Score gap (avg authentic - avg forged): {gap:.4f}")
    print(f"  Separation quality: {'EXCELLENT' if gap > 0.4 else 'GOOD' if gap > 0.25 else 'MODERATE' if gap > 0.1 else 'POOR'}")


---
## 8. Per-Agent Performance Analysis


In [ ]:
# Per-agent score comparison across all documents
agent_names = ["Visual Tamper Agent", "Text Forensics Agent", "Metadata & Provenance Agent"]

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for idx, agent_name in enumerate(agent_names):
    auth_agent_scores = []
    forg_agent_scores = []

    for r in authentic_reports:
        for ar in r.agent_results:
            if ar.agent_name == agent_name:
                auth_agent_scores.append(ar.score)

    for r in forged_reports:
        for ar in r.agent_results:
            if ar.agent_name == agent_name:
                forg_agent_scores.append(ar.score)

    x = np.arange(2)
    auth_mean = np.mean(auth_agent_scores) if auth_agent_scores else 0
    forg_mean = np.mean(forg_agent_scores) if forg_agent_scores else 0

    bars = axes[idx].bar(x, [auth_mean, forg_mean],
                         color=["#2ecc71", "#e74c3c"], width=0.5)
    axes[idx].set_xticks(x)
    axes[idx].set_xticklabels(["Authentic", "Forged"])
    axes[idx].set_ylim(0, 1)
    axes[idx].set_ylabel("Avg Score")
    axes[idx].set_title(agent_name, fontsize=11, fontweight="bold")

    for bar, val in zip(bars, [auth_mean, forg_mean]):
        axes[idx].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
                       f"{val:.3f}", ha="center", fontweight="bold")

plt.suptitle("Per-Agent Average Scores: Authentic vs Forged", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()


---
## 9. Multi-Threshold ROC Analysis


In [ ]:
# Multi-threshold analysis
thresholds = np.arange(0.1, 1.0, 0.05)
accuracies, precisions, recalls, f1s = [], [], [], []
fprs, tprs = [], []

for t in thresholds:
    tp = sum(1 for r in summary.results if r.label == "forged" and r.dis_score < t)
    tn = sum(1 for r in summary.results if r.label == "authentic" and r.dis_score >= t)
    fp = sum(1 for r in summary.results if r.label == "authentic" and r.dis_score < t)
    fn = sum(1 for r in summary.results if r.label == "forged" and r.dis_score >= t)

    total = tp + tn + fp + fn
    acc = (tp + tn) / total if total > 0 else 0
    prec = tp / (tp + fp) if (tp + fp) > 0 else 0
    rec = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1 = 2 * prec * rec / (prec + rec) if (prec + rec) > 0 else 0
    fpr = fp / (fp + tn) if (fp + tn) > 0 else 0
    tpr = tp / (tp + fn) if (tp + fn) > 0 else 0

    accuracies.append(acc)
    precisions.append(prec)
    recalls.append(rec)
    f1s.append(f1)
    fprs.append(fpr)
    tprs.append(tpr)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Metrics vs Threshold
axes[0].plot(thresholds, accuracies, 'b-', linewidth=2, label='Accuracy')
axes[0].plot(thresholds, precisions, 'g-', linewidth=2, label='Precision')
axes[0].plot(thresholds, recalls, 'r-', linewidth=2, label='Recall')
axes[0].plot(thresholds, f1s, 'm-', linewidth=2, label='F1 Score')
axes[0].axvline(x=0.65, color='orange', linestyle='--', label='Default threshold')
best_f1_idx = np.argmax(f1s)
axes[0].axvline(x=thresholds[best_f1_idx], color='cyan', linestyle=':', label=f'Best F1 ({thresholds[best_f1_idx]:.2f})')
axes[0].set_xlabel('Threshold', fontsize=12)
axes[0].set_ylabel('Score', fontsize=12)
axes[0].set_title('Metrics vs Threshold', fontsize=13, fontweight='bold')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# ROC Curve - sort by FPR for proper curve
sorted_pairs = sorted(zip(fprs, tprs))
sorted_fprs = [p[0] for p in sorted_pairs]
sorted_tprs = [p[1] for p in sorted_pairs]

axes[1].plot(sorted_fprs, sorted_tprs, 'b-', linewidth=2)
axes[1].plot([0, 1], [0, 1], 'k--', alpha=0.5, label='Random')
axes[1].set_xlabel('False Positive Rate', fontsize=12)
axes[1].set_ylabel('True Positive Rate', fontsize=12)
axes[1].set_title('ROC Curve', fontsize=13, fontweight='bold')

# AUC via trapezoidal rule
auc = np.trapz(sorted_tprs, sorted_fprs)
axes[1].legend([f'CertusDoc (AUC={abs(auc):.3f})', 'Random'])
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\nOptimal threshold: {thresholds[best_f1_idx]:.2f} (F1 = {f1s[best_f1_idx]:.4f})")
print(f"Approximate AUC: {abs(auc):.4f}")


---
## 10. Processing Time Analysis


In [ ]:
# Processing time breakdown
all_eval_reports = authentic_reports + forged_reports
if all_eval_reports:
    total_times = [r.processing_time_ms for r in all_eval_reports]

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Per-document processing time
    doc_names = [r.document.file_name[:20] for r in all_eval_reports]
    doc_colors = ["#2ecc71" if r in authentic_reports else "#e74c3c" for r in all_eval_reports]
    axes[0].barh(range(len(doc_names)), total_times, color=doc_colors)
    axes[0].set_yticks(range(len(doc_names)))
    axes[0].set_yticklabels(doc_names, fontsize=8)
    axes[0].set_xlabel("Processing Time (ms)", fontsize=12)
    axes[0].set_title("Per-Document Processing Time", fontsize=13, fontweight="bold")
    for i, t in enumerate(total_times):
        axes[0].text(t + 20, i, f"{t:.0f}ms", va="center", fontsize=8)

    # Per-agent average time
    agent_times = {}
    for r in all_eval_reports:
        for ar in r.agent_results:
            agent_times.setdefault(ar.agent_name, []).append(ar.processing_time_ms)

    agent_labels = list(agent_times.keys())
    agent_avg_times = [np.mean(agent_times[a]) for a in agent_labels]
    agent_bar_colors = ["#3498db", "#9b59b6", "#1abc9c"][:len(agent_labels)]

    bars = axes[1].bar(range(len(agent_labels)), agent_avg_times, color=agent_bar_colors)
    axes[1].set_xticks(range(len(agent_labels)))
    axes[1].set_xticklabels([a.replace(" Agent", "").replace(" & ", "\n& ") for a in agent_labels], fontsize=9)
    axes[1].set_ylabel("Avg Time (ms)", fontsize=12)
    axes[1].set_title("Per-Agent Average Processing Time", fontsize=13, fontweight="bold")
    for bar, val in zip(bars, agent_avg_times):
        axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5,
                     f"{val:.0f}ms", ha="center", fontweight="bold", fontsize=10)

    plt.tight_layout()
    plt.show()

    print(f"\nProcessing Time Summary:")
    print(f"  Total documents analyzed: {len(all_eval_reports)}")
    print(f"  Average per document: {np.mean(total_times):.0f}ms ({np.mean(total_times)/1000:.1f}s)")
    print(f"  Min: {min(total_times):.0f}ms | Max: {max(total_times):.0f}ms")
    print(f"  Throughput: ~{60000/np.mean(total_times):.1f} documents/minute")
    for a in agent_labels:
        print(f"  {a}: avg={np.mean(agent_times[a]):.0f}ms")


---
## 11. Summary Results Table


In [ ]:
# Complete results table
html = """
<style>
.results-table { border-collapse: collapse; width: 100%; font-family: monospace; font-size: 12px; }
.results-table th { background: #1a1a2e; color: #0ff; padding: 10px; text-align: left; }
.results-table td { padding: 8px; border-bottom: 1px solid #333; }
.results-table tr:hover { background: #1a1a2e; }
.score-high { color: #2ecc71; font-weight: bold; }
.score-med { color: #f39c12; font-weight: bold; }
.score-low { color: #e74c3c; font-weight: bold; }
.correct { color: #2ecc71; }
.wrong { color: #e74c3c; font-weight: bold; }
</style>
<table class="results-table">
<tr><th>Document</th><th>Label</th><th>DIS</th><th>Risk</th><th>Visual</th><th>Text</th><th>Metadata</th><th>Forgery Type</th><th>Correct?</th></tr>
"""

for r in summary.results:
    fname = Path(r.file_path).name[:35]
    score_class = "score-high" if r.dis_score >= 0.7 else "score-med" if r.dis_score >= 0.4 else "score-low"
    correct_class = "correct" if r.correct else "wrong"
    correct_text = "YES" if r.correct else "MISS"

    visual = r.agent_scores.get("Visual Tamper Agent", "-")
    text = r.agent_scores.get("Text Forensics Agent", "-")
    meta = r.agent_scores.get("Metadata & Provenance Agent", "-")

    visual_str = f"{visual:.4f}" if isinstance(visual, float) else visual
    text_str = f"{text:.4f}" if isinstance(text, float) else text
    meta_str = f"{meta:.4f}" if isinstance(meta, float) else meta

    label_color = "#2ecc71" if r.label == "authentic" else "#e74c3c"

    html += f'<tr>'
    html += f'<td>{fname}</td>'
    html += f'<td style="color:{label_color}">{r.label}</td>'
    html += f'<td class="{score_class}">{r.dis_score:.4f}</td>'
    html += f'<td>{r.risk_level}</td>'
    html += f'<td>{visual_str}</td>'
    html += f'<td>{text_str}</td>'
    html += f'<td>{meta_str}</td>'
    html += f'<td>{r.forgery_type}</td>'
    html += f'<td class="{correct_class}">{correct_text}</td></tr>'

html += '</table>'
display(HTML(html))


---
## 12. Generate Forensic PDF Reports


In [ ]:
# Generate PDF reports for key documents
from certusdoc.report.generator import generate_report
from IPython.display import FileLink

os.makedirs("reports", exist_ok=True)

for report in (authentic_reports + forged_reports):
    try:
        pdf_bytes = generate_report(report)
        fname = f"reports/certusdoc_{report.document.file_name}.pdf"
        with open(fname, "wb") as f:
            f.write(pdf_bytes)
        print(f"Generated: {fname} ({len(pdf_bytes)/1024:.1f} KB)")
    except Exception as e:
        print(f"Failed for {report.document.file_name}: {e}")

print("\nDownload reports from the reports/ directory.")


---
## 13. Upload Your Own Document for Testing
Upload any PDF or image to test it against CertusDoc.


In [ ]:
from google.colab import files

print("Upload a document (PDF, PNG, JPG, TIFF, BMP):")
uploaded = files.upload()

for filename, data in uploaded.items():
    # Save uploaded file
    with open(filename, "wb") as f:
        f.write(data)

    print(f"\nAnalyzing: {filename} ({len(data)/1024:.1f} KB)")
    print("=" * 60)

    report = pipeline.analyze(filename)

    visualize_result(report, title=f"YOUR DOCUMENT: {filename}")
    print_agent_breakdown(report)

    print(f"\nDIS Score: {report.dis_score:.4f}")
    print(f"Risk Level: {report.risk_level.value}")
    print(f"Recommendation: {report.recommended_action}")

    # Generate PDF report
    try:
        pdf_bytes = generate_report(report)
        report_name = f"certusdoc_report_{filename}.pdf"
        with open(report_name, "wb") as f:
            f.write(pdf_bytes)
        print(f"\nForensic report saved: {report_name}")
        files.download(report_name)
    except Exception as e:
        print(f"Report generation failed: {e}")


---
## 14. Architecture & Key Takeaways

### Pipeline Architecture
```
Document (PDF/Image)
    |
    v
[Stage 1: Ingestion]
    PDF rendering (300 DPI) -> Page images
    Tesseract OCR -> Text + confidence
    pikepdf/EXIF -> Metadata extraction
    |
    v
[Stage 2: Multi-Agent Detection] (parallel)
    |-- Visual Tamper Agent
    |   |-- ManTraNet (deep learning pixel-level forgery detection)
    |   |-- Multi-scale ELA (Q90/Q75/Q50)
    |   |-- Copy-move detection (ORB features)
    |   |-- JPEG quantization analysis
    |   |-- Noise consistency analysis
    |   |-- Print-scan detection (FFT halftone, moire)
    |   '-> Score + Heatmap
    |
    |-- Text Forensics Agent
    |   |-- OCR confidence analysis
    |   |-- Font consistency checking
    |   |-- Baseline alignment analysis
    |   |-- Spatial clustering of anomalies
    |   '-> Score + Findings
    |
    '-- Metadata & Provenance Agent
        |-- PDF metadata rules
        |-- EXIF analysis
        |-- QR code validation
        |-- Aadhaar checksum (Verhoeff)
        |-- Indian document pattern matching
        |-- Isolation Forest anomaly scoring
        '-> Score + Findings
    |
    v
[Stage 3: Weighted Trust Fusion]
    DIS = sum(Ri x Si) / sum(Ri)
    + Dynamic weight adjustment
    + Evidence-based scoring
    + Government provenance override
    |
    v
[Stage 4: Forensic Report]
    PDF report + Heatmap overlay + Risk classification
```

### Key Technical Features

| Feature | Description |
|---------|-------------|
| **ManTraNet** | Deep learning pixel-level forgery localization (pre-trained, no fine-tuning) |
| **Multi-scale ELA** | Error Level Analysis at Q90/Q75/Q50 for JPEG recompression detection |
| **Government Provenance Override** | Legitimate e-documents (wkhtmltopdf, reportlab) get a DIS floor of 0.75 |
| **Source-Aware Thresholds** | Different tolerance levels for known PDF generators |
| **Indian Document Specialization** | Aadhaar Verhoeff checksum, PAN, Passport, Voter ID, Bank Statement patterns |
| **QR Code Validation** | cv2.QRCodeDetector fallback when pyzbar unavailable |
| **Print-Scan Detection** | FFT halftone, Laplacian ink bleed, moire pattern analysis |
| **WhatsApp Detection** | Identifies messaging app forwarded documents (size/metadata heuristics) |
| **Weighted Trust Fusion** | Reliability-weighted agent score combination with evidence-based adjustments |
| **Forensic PDF Report** | Professional report with heatmap overlays and per-agent findings |


In [ ]:
# Final summary card
if summary and all_eval_reports:
    total_time = sum(r.processing_time_ms for r in all_eval_reports)
    avg_time = np.mean([r.processing_time_ms for r in all_eval_reports])

    display(HTML(f"""
    <div style="background: linear-gradient(135deg, #0d1117 0%, #1a1a2e 100%); color:#c9d1d9; padding:30px; border-radius:15px; font-family:monospace; margin-top:20px;">
    <h2 style="color:#58a6ff; text-align:center; margin-bottom:20px;">CertusDoc - Final Evaluation Summary</h2>
    <div style="display:flex; justify-content:space-around; flex-wrap:wrap;">
        <div style="text-align:center; padding:15px; min-width:120px;">
            <div style="font-size:36px; font-weight:bold; color:#2ecc71;">{summary.accuracy:.0%}</div>
            <div style="font-size:12px; color:#8b949e;">ACCURACY</div>
        </div>
        <div style="text-align:center; padding:15px; min-width:120px;">
            <div style="font-size:36px; font-weight:bold; color:#3498db;">{summary.f1:.3f}</div>
            <div style="font-size:12px; color:#8b949e;">F1 SCORE</div>
        </div>
        <div style="text-align:center; padding:15px; min-width:120px;">
            <div style="font-size:36px; font-weight:bold; color:#2ecc71;">{summary.precision:.0%}</div>
            <div style="font-size:12px; color:#8b949e;">PRECISION</div>
        </div>
        <div style="text-align:center; padding:15px; min-width:120px;">
            <div style="font-size:36px; font-weight:bold; color:#e67e22;">{summary.recall:.0%}</div>
            <div style="font-size:12px; color:#8b949e;">RECALL</div>
        </div>
        <div style="text-align:center; padding:15px; min-width:120px;">
            <div style="font-size:36px; font-weight:bold; color:#9b59b6;">{avg_time:.0f}ms</div>
            <div style="font-size:12px; color:#8b949e;">AVG TIME/DOC</div>
        </div>
    </div>
    <hr style="border-color:#30363d; margin:20px 0;">
    <div style="display:flex; justify-content:space-around; flex-wrap:wrap;">
        <div style="text-align:center; padding:10px;">
            <div style="font-size:20px; color:#2ecc71;">Avg Authentic DIS: <b>{summary.avg_authentic_score:.4f}</b></div>
        </div>
        <div style="text-align:center; padding:10px;">
            <div style="font-size:20px; color:#e74c3c;">Avg Forged DIS: <b>{summary.avg_forged_score:.4f}</b></div>
        </div>
        <div style="text-align:center; padding:10px;">
            <div style="font-size:20px; color:#f39c12;">Score Gap: <b>{summary.avg_authentic_score - summary.avg_forged_score:.4f}</b></div>
        </div>
    </div>
    <div style="text-align:center; margin-top:15px; font-size:11px; color:#8b949e;">
        Evaluated {summary.total} documents ({summary.authentic_count} authentic, {summary.forged_count} forged) | Threshold: 0.65
    </div>
    </div>
    """))
else:
    print("Run the evaluation cells above first.")
